## Nastavenie CW-Lite s XMEGA cieľom
Po zapojení CW do PC sa pripojíme pomocou prikazov
(Scope - zariadenie pomocou ktorého útočíme)

In [ ]:
import chipwhisperer as cw
scope = cw.scope()

Štandardne sa CW pokúsi detekovať platformu ktorou útočime (CW-Lite, CW-Nano atď), pre manuálnu špecifikáciu si pozri API dokumetáciu. 
V prípade že máš zapojených viacero CW zariadení musíš špecifikovať s ktorým, zariadením chceš pracovať

```python
scope = cw.scope(sn='<seriové číslo>')
```

Viac info [API sekcia na readthedocs](https://chipwhisperer.readthedocs.io/en/latest/scope-api.html).

Pripojenie na cieľové zariadenie:
(target - zariadenie na ktoré útočíme)

In [ ]:
target = cw.target(scope, cw.targets.SimpleSerial)

Niektoré rozumné predvolené nastavenia je možné nastaviť pomocou:

In [ ]:
scope.default_setup()

čo, ako je možné vidieť z dokumentácie, v prípade CWLite/CW1200 znamená nasledovné:

- Nastaví zosilnenie osciloskopu na 45 dB
- Nastaví osciloskop na zachytenie 5000 vzoriek
- Nastaví posun osciloskopu na 0 (t. j. začne zachytávať hneď po spustení)
- Nastaví trigger osciloskopu na stúpajúcu hranu
- Vysiela 7,37 MHz takt na cieľ na HS2
- Taktuje ADC osciloskopu na 4*7,37 MHz. Poznámka: toto je synchronizované s cieľovým taktom na HS2
- Priradí GPIO1 ako sériový RX
- Priradí GPIO2 ako sériový TX


A toto je všetkp, CW je pripravené na vykonávanie útokov

## Kompilácia firmvéru
Ak chceme vykonávať nejaký útok, najskôr potrebujeme na cieľ nahrať nejaký softvér na ktorý budeme útočiť
Pomocou jupyteru vieme kompilovať firmvér následovne


In [ ]:
%%bash
cd ../firmware/mcu/simpleserial-base/
make PLATFORM=CWLITEXMEGA CRYPTO_TARGET=NONE

Keďže sme nechali PLATFORM prázdne mal by sa zobraziť list PLATFORM pre ktoré môžeme kompilovať. Keďže toto je návod pre CW-Lite s cieľom XMEGA tak použi `CWLITEXMEGA` 

Ak sa ti podarilo úspešne skompilovať program následne ho treba nahrat pomocou bootloaderu nahrať. 

In [ ]:
cw.program_target(scope, cw.programmers.XMEGAProgrammer, "cesta/k/firmveru.hex")

<details>
<summary>Nápoveda</summary>

Cesta k programu: <br>
"../firmware/mcu/simpleserial-base/simpleserial-base-CWLITEXMEGA.hex"

</details>

## Komunikácia s cieľom
S cieľom komunikujeme pomocou `target` objektu ktorý sme si definovali vyššie
Komunikovať vieme pomocou jednou z dvoch kategórií
1. Raw serial pomocou `target.read()`, `target.write()`, `target.flush()`, atď.
2. SimpleSerial príkazy pomocou `target.simpleserial_read()`, `target.simpleserial_write()`, `target.simpleserial_wait_ack()`, atď.

Firmvér, ktorý sme nahrali používa SimpleSeial protokol (https://wiki.newae.com/SimpleSerial), ktorým aj začneme. Neskôr skúsime aj Raw serial

Keď sa pozrieme bližsie na simpleserial-base firmvér (simpleserial-base.c), tak zistíme že pomocou simpleserial príkazu `'p'` nám cieľ vráti naspäť čo sme mu poslali

<details>
<summary>simpleserial-base.c - funkcia p</summary>

    <pre><code>
    uint8_t get_pt(uint8_t* pt, uint8_t len)
    {
        /**********************************
        * Start user-specific code here. */
        trigger_high();

        //16 hex bytes held in 'pt' were sent
        //from the computer. Store your response
        //back into 'pt', which will send 16 bytes
        //back to computer. Can ignore of course if
        //not needed

        trigger_low();
        /* End user-specific code here. *
        ********************************/
        simpleserial_put('r', 16, pt);
        return 0x00;
    } 
    </code></pre>

</details>

Vyskúšaj si to:

In [ ]:
msg = bytearray([0]*16) #simpleserial používa bytearrays
target.simpleserial_write('p', msg)

Skontroluj či si dostal odpoveď

In [ ]:
print(target.simpleserial_read('r', 16))

Náš program pozná okrem `'p'`, `'r'`, aj `'k'`.
Skús ho

<details>
<summary>Riešenie</summary>
<pre><code>
msg = bytearray([0]*16) 
target.simpleserial_write('k', msg)
</code></pre>
</details>

Tento príkaz nám síce nič nevráti, mal by však potvrdiť prijatie a vrátiť chybovú správu:

In [ ]:
print(target.simpleserial_wait_ack()) #mal by vrátiť 0

Simpleserial správy majú zvyčajne tvar:
`príkazový_znak + ascii_encoded_byty + '\n'`
V našom prvom príklade `príkazový_znak = 'p'` a  `ascii_encoded_byty = "00"*16`
(pozn. 00 != 0x00, v ascii "00" ma hex hodnotu 0x3030)
Skús poslať príkaz ale namiesto `target.simpleserial_write()` použi `target.write()`

In [ ]:
target.write('p' + ) #Doplň zvyšok

target.read() nám umožní vráti všetky znaky ktoré nám poslal cieľ

In [ ]:
recv_msg = ""

In [ ]:
recv_msg += target.read() #Može sa stať že to budeš musieť volať viac krát pre celú správu
print(recv_msg)

Od verzie ChipWhisperer 5.4 je k dispozícii nový komunikačný protokol pre cieľové zariadenie s názvom SimpleSerial 2. V porovnaní so starým protokolom SimpleSerial má viacero výhod:

1. Data is unencoded binary (except for byte stuffing) instead of ACII encoded, meaning far more data can be sent per packet (just under double)
2. It's got a command and subcommand field that get passed to the target callback functions, meaning more can be done per packet. For example, the simpleserial-aes firmware has been modified to allow plaintext, key, and/or mask to be set with a single packet.
3. It's got a length field, meaning the same target commands can take different length packets depending on the situation. This, for example, allows both plaintext and key (or only one) to be send with the same command in simpleserial-aes.
4. It's got an 8-bit CRC (0xA6) for data integrety.
5. Frames are consistant overhead byte stuffed (COBS). This makes it easy to reset communication (done simply by sending two 0x00 bytes) and helps catch malformed length bytes.

## Zachytávanie priebehov

Teraz, keď je cieľ naprogramovaný a vieme, ako s ním komunikovať, začnime zaznamenávať spotrebu energie! Postup zachytávania:

1. Aktivujte ChipWhisperer pomocou `scope.arm()`. Zachytávanie sa spustí hneď po aktivácii (v našom prípade ide o stúpajúcu hranu na `gpio4`).
1. `scope.capture()` prečíta zaznamenanú stopu napájania a zablokuje sa, kým ChipWhisperer nedokončí nahrávanie alebo kým nedôjde k časovému limitu osciloskopu.  Táto funkcia ale nevráti zaznamenané údaje z osciloskopu ale chybu ak k nej došlo.
1. Zachytenú krivku napájania môžete prečítať pomocou `scope.get_last_trace()`.

`simpleserial_base` spustí ChipWhisperer, keď pošleme príkaz `‚p‘`. Skús teraz zachytiť krivku:

In [ ]:
scope.arm()
target.simpleserial_write('p', msg)

## fill in the rest...

<details>
<summary>Riešenie</summary>
<pre><code>
scope.arm()
target.simpleserial_write('p', msg)
scope.capture()
print(scope.get_last_trace())
</code></pre>
</details>

## Záver

Úvod máme za sebou, teraz môžeš pokračovať s nejakou konkrétnou lekciou napr.(fault0_1 alebo sca0_1)

Na konci ešte nezabudni odpojiť tento "notebook" od chipwhisperera

In [ ]:
scope.dis()
target.dis()